### 동시출현행렬
- 특정 단어를 기준으로 주변의 범위 안에서 어떠한 단어가 등장했는가? 빈도 수를 확인하는 행렬
- ex)
    - '오늘 날씨가 너무 좋다', '내알 날씨가 조금 흐리다'
    - window(범위)가 1인 경우
    - 날씨가 -> 주변의 단어들을 ['오늘', '너무', '내일', '조금'
- N-gram 차이
    - N-gram : 단어 순서 반영
    - 동시출현행렬 : 의미 관계를 반영

### PMI
- 두 단어가 우연히 함께 등장한 것인가? 아니면 의미적으로 연관성이 있는가? 측정
- 측정 값이 클수록 의미적으로 강하게 연결되어있다는 의미

### PPMI
- PMI가 음수인 경우는 사용되는 경우가 극히 드물
- PMI의 음수의 데이터는 0으로 대체하는 값

In [1]:
import math
from collections import Counter
import pandas as pd
from konlpy.tag import Okt

In [2]:
docs = [
    '오늘 날씨가 매우 좋다',
    '오늘 기분이 정말 좋다',
    '내일 날씨가 조금 흐리다',
    '기분이 매우 나쁘다'
]

In [3]:
# 좌우 단어 검색의 범위를 지정
window_size = 1


In [5]:
# 토큰화 함수를 정의
okt = Okt()

def tokenize(text):
    result = okt.morphs(text)
    return result

tokens = [tokenize(doc) for doc in docs]
tokens

[['오늘', '날씨', '가', '매우', '좋다'],
 ['오늘', '기분', '이', '정말', '좋다'],
 ['내일', '날씨', '가', '조금', '흐리다'],
 ['기분', '이', '매우', '나쁘다']]

In [10]:
# tokense데이터에서 나오는 단어들의 목록을 생성
# 중복 데이터를 제거하고 리스트의 형태로 변환
# 2차원 리스트를 1차원으로 변경 -> 집합의 형태로 변환(set())
vocab1 = sorted(set(sum(tokens, [])))

# vocab2 = list(set(sum(tokens, []))).sort()      # list.sort()를 이용하면 list class 객체 안에 변수를 변경(return None) 
vocab2 = list(set(sum(tokens, [])))
vocab2.sort()

In [11]:
print(vocab1)
print(vocab2)

['가', '기분', '나쁘다', '날씨', '내일', '매우', '오늘', '이', '정말', '조금', '좋다', '흐리다']
['가', '기분', '나쁘다', '날씨', '내일', '매우', '오늘', '이', '정말', '조금', '좋다', '흐리다']


In [12]:
# 단어들의 목록을 인덱스와 함께 dict 형태로 저장
vocab_index = {
    word : idx for idx, word in enumerate(vocab1)
}
vocab_index

{'가': 0,
 '기분': 1,
 '나쁘다': 2,
 '날씨': 3,
 '내일': 4,
 '매우': 5,
 '오늘': 6,
 '이': 7,
 '정말': 8,
 '조금': 9,
 '좋다': 10,
 '흐리다': 11}

In [19]:
# 동시출현 행렬 생성 -> 기본값으로 행렬을 먼저 생성
# len(vocab1) 만큼 행렬을 생성 -> 12 * 12 2차원 리스트 생성
co_metrix = [[0] * len(vocab1) for i in range(len(vocab1))]
co_metrix

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [ ]:
# window_size를 기반으로 하여 동시출현 카운트

for token in tokens:
    # token : tokens의 각각의 원소 (1차원 리스트)
    # print(token)
    for idx, word in enumerate(token):
        # idx : token에서 각각의 인덱스의 값
        # word : token에서 각각의 원소의 값
        # word의 인덱스의 값
        center_idx = vocab_index[word]
        start = max(0, idx - window_size)
        end = min(len(token), idx + window_size+1)     
        # range()함수에서 end를 종료값으로 이용하여 해당 값은 미포함
        # idx + window_size가 range에서 체크가 불가능하기 때문에 +1
        for i in range(start, end):
            # 동일한 데이터인 경우에는 카운터 증가x
            if idx != i:
                context = token[i]  # 단어 선택
                context_idx = vocab_index[context]  # 해당 단어의 인덱스 값
                # context_idx -> 동시출현 행렬의 idx의 값
                co_metrix[center_idx][context_idx] += 1
                

In [23]:
co_df = pd.DataFrame(co_metrix, index=vocab1, columns=vocab1)
co_df

,가,기분,나쁘다,날씨,내일,매우,오늘,이,정말,조금,좋다,흐리다
가,0,0,0,2,0,1,0,0,0,1,0,0
기분,0,0,0,0,0,0,1,2,0,0,0,0
나쁘다,0,0,0,0,0,1,0,0,0,0,0,0
날씨,2,0,0,0,1,0,1,0,0,0,0,0
내일,0,0,0,1,0,0,0,0,0,0,0,0
매우,1,0,1,0,0,0,0,1,0,0,1,0
오늘,0,1,0,1,0,0,0,0,0,0,0,0
이,0,2,0,0,0,1,0,0,1,0,0,0
정말,0,0,0,0,0,0,0,1,0,0,1,0
조금,1,0,0,0,0,0,0,0,0,0,0,1


In [37]:
# PMI & PPMI를 계산

# co_df의 value 총 합계
sum(sum(co_df.values))
total_count = sum(sum(row) for row in co_metrix)

In [38]:
# 각 단어별 동시 등장 횟수의 합계 / total_count
p_word = [sum(row) / total_count for row in co_metrix]
p_word


[0.13333333333333333,
 0.1,
 0.03333333333333333,
 0.13333333333333333,
 0.03333333333333333,
 0.13333333333333333,
 0.06666666666666667,
 0.13333333333333333,
 0.06666666666666667,
 0.06666666666666667,
 0.06666666666666667,
 0.03333333333333333]

In [ ]:
p_context = [
    sum(
        co_metrix[i][j] for i in range(len(vocab1))) / total_count for j in range(len(vocab1))
]
p_context

[0.13333333333333333,
 0.1,
 0.03333333333333333,
 0.13333333333333333,
 0.03333333333333333,
 0.13333333333333333,
 0.06666666666666667,
 0.13333333333333333,
 0.06666666666666667,
 0.06666666666666667,
 0.06666666666666667,
 0.03333333333333333]

In [46]:
# PMI 계산식
def calc_pmi(i, j):
    p_wc = co_metrix[i][j] / total_count
    if p_wc == 0:
        result =  0
    else:
        result = math.log2(p_wc / (p_word[i] * p_context[j]) + 1e-12)
    return result

pmi_metrix = [ [calc_pmi(i, j) for j in range(len(vocab1))] for i in range(len(vocab1))]

pmi_df = pd.DataFrame(pmi_metrix, index=vocab1, columns=vocab1)
pmi_df.style.background_gradient(cmap='Blues')

,가,기분,나쁘다,날씨,내일,매우,오늘,이,정말,조금,좋다,흐리다
가,0.000000,0.000000,0.000000,1.906891,0.000000,0.906891,0.000000,0.000000,0.000000,1.906891,0.000000,0.000000
기분,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.321928,2.321928,0.000000,0.000000,0.000000,0.000000
나쁘다,0.000000,0.000000,0.000000,0.000000,0.000000,2.906891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
날씨,1.906891,0.000000,0.000000,0.000000,2.906891,0.000000,1.906891,0.000000,0.000000,0.000000,0.000000,0.000000
내일,0.000000,0.000000,0.000000,2.906891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
매우,0.906891,0.000000,2.906891,0.000000,0.000000,0.000000,0.000000,0.906891,0.000000,0.000000,1.906891,0.000000
오늘,0.000000,2.321928,0.000000,1.906891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
이,0.000000,2.321928,0.000000,0.000000,0.000000,0.906891,0.000000,0.000000,1.906891,0.000000,0.000000,0.000000
정말,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.906891,0.000000,0.000000,2.906891,0.000000
조금,1.906891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.906891
